# 点云分类系统演示
## 基于点云数据的室内场景三维物体分类

这个Notebook演示了如何使用点云分类系统进行数据下载、模型训练和评估。

## 1. 环境设置

In [ ]:
# 导入必要的库
import os
import sys
import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# 添加项目根目录到Python路径
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

print(f"项目根目录: {project_root}")

## 2. 下载ScanObjectNN数据集

In [ ]:
# 使用系统命令下载数据集（如果未下载）
data_dir = project_root / "data" / "scanobjectnn"
if not data_dir.exists():
    print("下载ScanObjectNN数据集...")
    !cd {project_root} && python main.py download-scanobjectnn --data_dir {data_dir}
else:
    print(f"数据集已存在: {data_dir}")

In [ ]:
# 查看数据集统计信息
stats_file = data_dir / "dataset_statistics.json"
if stats_file.exists():
    with open(stats_file, 'r', encoding='utf-8') as f:
        stats = json.load(f)
    
    print("数据集统计信息:")
    print(f"  样本数量: {stats.get('num_samples', 0)}")
    print(f"  类别数量: {stats.get('num_classes', 0)}")
    print(f"  类别名称: {stats.get('class_names', [])}")
    
    # 可视化类别分布
    class_dist = stats.get('class_distribution', {})
    if class_dist:
        plt.figure(figsize=(12, 6))
        plt.bar(class_dist.keys(), class_dist.values())
        plt.xlabel('类别')
        plt.ylabel('样本数量')
        plt.title('ScanObjectNN数据集类别分布')
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()

## 3. 可视化点云样本

In [ ]:
# 可视化几个样本
print("可视化点云样本...")
!cd {project_root} && python main.py visualize --dataset scanobjectnn --data_dir {data_dir} --num_samples 3 --output_dir ./notebooks/visualizations

## 4. 创建和比较模型

In [ ]:
# 比较不同模型
print("比较不同模型...")
!cd {project_root} && python main.py compare --models point_transformer,pointnet,dgcnn --output ./notebooks/model_comparison.json

In [ ]:
# 加载比较结果
comparison_file = project_root / "notebooks" / "model_comparison.json"
if comparison_file.exists():
    with open(comparison_file, 'r', encoding='utf-8') as f:
        comparison = json.load(f)
    
    print("模型比较结果:")
    for model_name, result in comparison.items():
        print(f"\n{model_name}:")
        print(f"  状态: {result['status']}")
        if result['status'] == 'success':
            print(f"  参数数量: {result['total_params']:,}")
            print(f"  输出形状: {result['output_shape']}")

## 5. 训练Point Transformer模型

In [ ]:
# 训练模型（快速演示，只训练少量epoch）
print("训练Point Transformer模型...")
!cd {project_root} && python scripts/train.py --experiment demo_train --epochs 10 --batch_size 8 --learning_rate 0.001 --data_dir {data_dir} --model point_transformer --dataset scanobjectnn

## 6. 评估训练好的模型

In [ ]:
# 评估模型
checkpoint_path = project_root / "checkpoints" / "demo_train" / "best_model.pth"
if checkpoint_path.exists():
    print("评估训练好的模型...")
    !cd {project_root} && python main.py evaluate --checkpoint {checkpoint_path} --dataset scanobjectnn --data_dir {data_dir}

## 7. 分析训练结果

In [ ]:
# 加载训练历史
history_file = project_root / "checkpoints" / "demo_train" / "training_history.json"
if history_file.exists():
    with open(history_file, 'r', encoding='utf-8') as f:
        history = json.load(f)
    
    # 绘制训练曲线
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    # 损失曲线
    axes[0].plot(history.get('train_loss', []), 'b-', label='训练损失')
    axes[0].plot(history.get('val_loss', []), 'r-', label='验证损失')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('损失')
    axes[0].set_title('训练和验证损失')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # 准确率曲线
    axes[1].plot(history.get('train_accuracy', []), 'b-', label='训练准确率')
    axes[1].plot(history.get('val_accuracy', []), 'r-', label='验证准确率')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('准确率')
    axes[1].set_title('训练和验证准确率')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # 显示最佳结果
    if history.get('val_accuracy'):
        best_acc = max(history['val_accuracy'])
        best_epoch = history['val_accuracy'].index(best_acc) + 1
        print(f"最佳验证准确率: {best_acc:.4f} (Epoch {best_epoch})")

## 8. 在Kaggle上训练（可选）

In [ ]:
# Kaggle环境设置（如果在Kaggle中运行）
if 'kaggle' in str(project_root).lower():
    print("检测到Kaggle环境，设置Kaggle优化...")
    !python scripts/setup_kaggle.py
    
    # Kaggle优化训练
    print("在Kaggle上训练模型...")
    !cd {project_root} && python scripts/train.py --experiment kaggle_demo --epochs 50 --kaggle

## 9. 结果分析

In [ ]:
# 分析实验结果
print("分析实验结果...")
!cd {project_root} && python -m evaluation.analyzer --experiment_dir ./checkpoints --output_dir ./notebooks/analysis

## 10. 创建自定义配置

In [ ]:
# 创建自定义配置文件
print("创建自定义配置文件...")
!cd {project_root} && python main.py create-config --project_name "My Point Cloud Project" --model point_transformer --dataset scanobjectnn --output ./notebooks/my_config.yaml

## 总结

这个演示展示了点云分类系统的主要功能：

1. **数据下载**: 下载和准备ScanObjectNN数据集
2. **数据可视化**: 查看点云样本和类别分布
3. **模型比较**: 比较不同模型的参数和结构
4. **模型训练**: 训练Point Transformer模型
5. **模型评估**: 评估训练好的模型性能
6. **结果分析**: 分析训练过程和结果
7. **Kaggle集成**: 在Kaggle环境中训练
8. **配置管理**: 创建自定义配置文件

你可以修改参数和配置来探索不同的模型和训练策略。